## Selective Copying Task

In [ ]:
# Generates B shuffled batches of length seq_len with data_len tokens of interest from (0, vocab), and remaining noise tokens 0
def generate_batch(B, seq_len = 4096, data_len = 16, vocab = 16, device = dev):
    # generate random indices for data tokens, shuffle into noise tokens
    pos = torch.rand(B, seq_len, device = device).argsort(dim = 1)[:, :data_len].sort(dim = 1).values
    data = torch.randint(1, vocab, (B, data_len), device = device)
    x = torch.zeros(B, seq_len + data_len, dtype = torch.long, device = device)
    x.scatter_(1, pos, data)

    x[:, seq_len:] = vocab
    return x, data


In [ ]:
class Model(nn.Module):
    def __init__(self, cell, n_layers, vocab = 16, dim = 64, expansion = 6, dropout = 0.1):
        super().__init__()

        # convert discrete tokens to vector i.e. factor() in R
        self.emb = nn.Embedding(vocab + 1, dim)
        # layer normalization - some type of normalization is used but unspecified
        self.norms = nn.ModuleList([nn.LayerNorm(dim) for _ in range(n_layers)])
        self.layers = nn.ModuleList([cell(dim, hidden_dim = dim * expansion, proj_out = True) for _ in range(n_layers)])
        # dropout of 0.1 specified
        self.drop = nn.Dropout(dropout)
        self.fnorm = nn.LayerNorm(dim) # final layernorm
        self.head = nn.Linear(dim, vocab)

    def forward(self, x):
        h = self.emb(x)
        for norm, layer in zip(self.norms, self.layers):
            # residual connection specified
            h = h + self.drop(layer(norm(h)))
        return self.head(self.fnorm(h))


def train(cell, n_layers, steps, seq_len = 4096, batch = 64, accum = 2, lr = 3e-4, clip = 1.0):
    torch.manual_seed(0)
    model = Model(cell, n_layers).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr = lr)
    t0 = time.time()

    for step in range(1, steps + 1):
        opt.zero_grad(set_to_none = True)
        # gradient accumulation, split into smaller sub-batches for memory
        for a in range(accum):
            x, y = generate_batch(batch // accum, seq_len = seq_len, device = dev)
            # predict and keep only outputs
            y_hat = model(x)[:, -y.size(1):]

            loss = F.cross_entropy(y_hat.reshape(-1, 16), y.reshape(-1))
            (loss / accum).backward()
        # gradient clipping specified
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()

        if step % 1000 == 0:
            print(f"Step {step}/{steps} | Loss: {loss.item():.4f} | {(time.time() - t0) / step:.2f}s/step")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for _ in range(8):
            x, y = generate_batch(batch, seq_len = seq_len, device = dev)
            y_hat = model(x)[:, -y.size(1):].argmax(dim = -1)
            correct += (y_hat == y).sum().item()
            total += y.numel()
    return correct / total

In [ ]:
accuracy = train(minLSTM, 1, steps = 400000)
print(f"minLSTM (1 layer), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minLSTM, 2, steps = 400000)
print(f"minLSTM (2 layers), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minLSTM, 3, steps = 400000)
print(f"minLSTM (3 layers), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minGRU, 1, steps = 400000)
print(f"minGRU (1 layer), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minGRU, 2, steps = 400000)
print(f"minGRU (2 layer), Accuracy: {accuracy * 100:.2f}%")

accuracy = train(minGRU, 3, steps = 400000)
print(f"minGRU (3 layer), Accuracy: {accuracy * 100:.2f}%")

KeyboardInterrupt: 